# Figure 3 — M8 pooling vs Fourier pooling localization convergence

This notebook generates **readme Figure 3**: a spatial comparison of **average pooling** vs **Fourier pooling** learning behaviour on the M8 single-view xyz localisation task.

It trains both branches on the **existing full M8 corpus** with the historical consensus learning rates from `GummyBearTomography_Final_Report.ipynb` / `M8_CANONICAL_LR_BY_ARCH` (`pooled=0.001`, `fourier=0.03`), logs predicted xyz on the **full filtered validation set** every epoch (including initialization), writes reusable JSON histories, then builds POV-Ray frames **only from those JSON files** (identity map: catalog millimetres = POV world, z-up).

Visual language: closer **face-on** translucent grey bear (same azimuth as the network-scene illustration, mostly +X); **green** (bright) validation targets; **deep red** average-pooling predictions + links; **dark blue** Fourier predictions + links (luminous, slightly dimmer than green).

Generated files go under gitignored `outputs/figure3_learning_convergence/` (not `figures/`).

Install from the repo root (no editable install): `pip install ".[dl]" -c requirements.txt`


In [1]:
from __future__ import annotations
from pathlib import Path


# Install the library from the local repo

# First get path. If for any reason, we are not at repo root, move up until "pyproject.toml" is found
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
# Knowing the repo root, we can install the libraries in the venv    
!pip install --quiet --no-cache-dir "{ROOT}[fem]" -c "{ROOT}/requirements.txt"

# If FEM package installation is difficult, you can try dl (deep learning) + dev tools (dev):
# !pip install --quiet --no-cache-dir "{ROOT}[dl,dev]" -c "{ROOT}/requirements.txt"
# In that case, keep the global DATA_MODE = "inspect" (set just after Imports).


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:

from gummybear.paths import display_path, repo_relative_path
from gummybear_illustration.figure3_export import export_figure3_convergence
from gummybear_illustration.figure3_history import (
    default_figure3_root,
    load_history,
    select_best_fourier_advantage_sample,
)
from gummybear_illustration.paths import repo_root
from tomography_ml.studies.single_view_m8 import M8_CANONICAL_LR_BY_ARCH

ROOT = repo_root()

# Gitignored: outputs/figure3_learning_convergence/ (JSON, POV, GIF, PNG).
OUTPUT_ROOT = default_figure3_root(ROOT)

# Full M8 protocol (matches final-report xyz study filters inside the package).
NUM_EPOCHS = 200
BATCH_SIZE = 32
SEED = 0
# None = entire filtered validation set (green targets + red/blue links).
N_TRACKED = None
DEVICE = None  # None → tomography_ml.get_device()

# If JSON histories already exist under OUTPUT_ROOT, skip retraining.
# Older runs wrote under figures/figure3_learning_convergence/; copy the
# prediction_history/ folder here, or set FORCE_RETRAIN=True once.
SKIP_TRAIN_IF_HISTORY_EXISTS = True
FORCE_RETRAIN = False

# Rendering (consumes JSON only; does not read live tensors).
RENDER = True
# Frame-level POV-Ray parallelism. True uses cpu_count-2 workers (at least 1).
RENDER_MULTIPROCESSING = True
# 1 = every epoch (full motion GIF). Increase for faster POV iteration.
RENDER_EPOCH_STRIDE = 10
GIF_DURATION_MS = 200

# POV camera in simulation millimetres (z-up). None = auto (network-scene
# azimuth, long standoff). Copy resolved values from the export printout
# to pin an exact view.
CAMERA_LOCATION = (0.0,70.0,8.0)  # e.g. (90.0, -16.0, 28.0)
CAMERA_LOOK_AT = (-1.1, 0.5, 6.5)
CAMERA_FOV_DEG = 42.0
# Scales the scene key/fill lights only; marker glow is controlled separately.
LIGHT_INTENSITY = 1.35
# Per-ball red/green/blue point lights (slow with the full validation set).
BALL_LIGHTS = False
# Green target emission (1/3 of the previous default).
GREEN_LUMINOSITY = 0
# Target→predicted cylinders (usually off).
DRAW_PREDICTION_LINKS = False
# Distance-based emission ramp on balls (usually off; use transparency instead).
DISTANCE_LUMINOSITY = True
# Distance-based transparency on predicted balls only (usually on).
DISTANCE_TRANSPARENCY = True
# Red emission when coincident with the green target (if DISTANCE_LUMINOSITY).
POOLING_LUMINOSITY_AT_TARGET = 0.18
# Blue emission when coincident with the green target (if DISTANCE_LUMINOSITY).
FOURIER_LUMINOSITY_AT_TARGET = 0.5
# Back-compat shared knob; leave unused unless you want one value for both.
PRED_LUMINOSITY_AT_TARGET = None
# Hold the on-target emission through this radius, then decay linearly.
PRED_LUMINOSITY_HOLD_MM = 1.0
# Emission reaches background at/beyond this distance.
PRED_LUMINOSITY_ZERO_MM = 1.5
# POV transmit when far (0=opaque, 1=invisible). On target uses PRED_TRANSMIT_AT_TARGET.
PRED_TRANSMIT_FAR = 0.9
PRED_TRANSMIT_AT_TARGET = 0.0
# Hold the on-target transparency through this radius, then decay linearly.
PRED_TRANSMIT_HOLD_MM = 2.0
# Transparency reaches PRED_TRANSMIT_FAR at/beyond this distance.
PRED_TRANSMIT_ZERO_MM = 5.0

print(f"ROOT={display_path(ROOT)}")
print(f"OUTPUT_ROOT={display_path(OUTPUT_ROOT)}")
print(f"M8_CANONICAL_LR_BY_ARCH={dict(M8_CANONICAL_LR_BY_ARCH)}")
print(
    f"NUM_EPOCHS={NUM_EPOCHS}  SEED={SEED}  N_TRACKED={N_TRACKED}  "
    f"SKIP_TRAIN_IF_HISTORY_EXISTS={SKIP_TRAIN_IF_HISTORY_EXISTS}  "
    f"RENDER={RENDER}  RENDER_MULTIPROCESSING={RENDER_MULTIPROCESSING}"
)
print(
    f"CAMERA_LOCATION={CAMERA_LOCATION}  CAMERA_LOOK_AT={CAMERA_LOOK_AT}  "
    f"CAMERA_FOV_DEG={CAMERA_FOV_DEG}  LIGHT_INTENSITY={LIGHT_INTENSITY}  "
    f"BALL_LIGHTS={BALL_LIGHTS}  GREEN_LUMINOSITY={GREEN_LUMINOSITY}  "
    f"DRAW_PREDICTION_LINKS={DRAW_PREDICTION_LINKS}  "
    f"DISTANCE_LUMINOSITY={DISTANCE_LUMINOSITY}  "
    f"DISTANCE_TRANSPARENCY={DISTANCE_TRANSPARENCY}  "
    f"POOLING_LUMINOSITY_AT_TARGET={POOLING_LUMINOSITY_AT_TARGET}  "
    f"FOURIER_LUMINOSITY_AT_TARGET={FOURIER_LUMINOSITY_AT_TARGET}  "
    f"PRED_LUMINOSITY_AT_TARGET={PRED_LUMINOSITY_AT_TARGET}  "
    f"PRED_LUMINOSITY_HOLD_MM={PRED_LUMINOSITY_HOLD_MM}  "
    f"PRED_LUMINOSITY_ZERO_MM={PRED_LUMINOSITY_ZERO_MM}  "
    f"PRED_TRANSMIT_FAR={PRED_TRANSMIT_FAR}  "
    f"PRED_TRANSMIT_AT_TARGET={PRED_TRANSMIT_AT_TARGET}  "
    f"PRED_TRANSMIT_HOLD_MM={PRED_TRANSMIT_HOLD_MM}  "
    f"PRED_TRANSMIT_ZERO_MM={PRED_TRANSMIT_ZERO_MM}"
)
print(
    "Coordinate convention: particle_x/y/z are simulation millimetres (z-up), "
    "identical to the STL / physical-scene POV world (identity transform)."
)


ROOT=.
OUTPUT_ROOT=outputs/figure3_learning_convergence
M8_CANONICAL_LR_BY_ARCH={'pooled': 0.001, 'fourier': 0.03, 'flatten': 0.0003}
NUM_EPOCHS=200  SEED=0  N_TRACKED=None  SKIP_TRAIN_IF_HISTORY_EXISTS=True  RENDER=True  RENDER_MULTIPROCESSING=True
CAMERA_LOCATION=(0.0, 70.0, 8.0)  CAMERA_LOOK_AT=(-1.1, 0.5, 6.5)  CAMERA_FOV_DEG=42.0  LIGHT_INTENSITY=1.35  BALL_LIGHTS=False  GREEN_LUMINOSITY=0  DRAW_PREDICTION_LINKS=False  DISTANCE_LUMINOSITY=True  DISTANCE_TRANSPARENCY=True  POOLING_LUMINOSITY_AT_TARGET=0.18  FOURIER_LUMINOSITY_AT_TARGET=0.5  PRED_LUMINOSITY_AT_TARGET=None  PRED_LUMINOSITY_HOLD_MM=1.0  PRED_LUMINOSITY_ZERO_MM=1.5  PRED_TRANSMIT_FAR=0.9  PRED_TRANSMIT_AT_TARGET=0.0  PRED_TRANSMIT_HOLD_MM=2.0  PRED_TRANSMIT_ZERO_MM=5.0
Coordinate convention: particle_x/y/z are simulation millimetres (z-up), identical to the STL / physical-scene POV world (identity transform).


## 1. Train (or reuse JSON) and export Figure 3

`export_figure3_convergence` is the only expensive cell. **Run All** is enough.

It trains pooled vs Fourier when needed, writes under `outputs/figure3_learning_convergence/`:

- `prediction_history/m8_pooling_history.json`
- `prediction_history/fourier_pooling_history.json`
- `prediction_history/combined_prediction_history.json`

then builds POV scenes from that JSON and, if `RENDER=True`, ray-traces frames, the GIF, and the final still. Training is skipped when histories already exist (`SKIP_TRAIN_IF_HISTORY_EXISTS=True`). Change camera or material knobs and Run All again to rebuild scenes/renders without retraining. Set `FORCE_RETRAIN=True` once if an older history only logged a subset of validation samples.


In [ ]:
result = export_figure3_convergence(
    repo_root_path=ROOT,
    output_root=OUTPUT_ROOT,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    seed=SEED,
    n_tracked=N_TRACKED,
    skip_train_if_history_exists=SKIP_TRAIN_IF_HISTORY_EXISTS,
    force_retrain=FORCE_RETRAIN,
    render=RENDER,
    multiprocessing=RENDER_MULTIPROCESSING,
    render_epoch_stride=RENDER_EPOCH_STRIDE,
    gif_duration_ms=GIF_DURATION_MS,
    device=DEVICE,
    camera_location=CAMERA_LOCATION,
    camera_look_at=CAMERA_LOOK_AT,
    fov_deg=CAMERA_FOV_DEG,
    light_intensity=LIGHT_INTENSITY,
    ball_lights=BALL_LIGHTS,
    draw_prediction_links=DRAW_PREDICTION_LINKS,
    distance_luminosity=DISTANCE_LUMINOSITY,
    distance_transparency=DISTANCE_TRANSPARENCY,
    green_luminosity=GREEN_LUMINOSITY,
    pred_luminosity_at_target=PRED_LUMINOSITY_AT_TARGET,
    pooling_luminosity_at_target=POOLING_LUMINOSITY_AT_TARGET,
    fourier_luminosity_at_target=FOURIER_LUMINOSITY_AT_TARGET,
    pred_luminosity_hold_mm=PRED_LUMINOSITY_HOLD_MM,
    pred_luminosity_zero_mm=PRED_LUMINOSITY_ZERO_MM,
    pred_transmit_far=PRED_TRANSMIT_FAR,
    pred_transmit_hold_mm=PRED_TRANSMIT_HOLD_MM,
    pred_transmit_zero_mm=PRED_TRANSMIT_ZERO_MM,
    pred_transmit_at_target=PRED_TRANSMIT_AT_TARGET,
    verbose=True,
)
layout = result["layout"]
summary = result["summary"]
for key in ("pooling", "fourier", "combined", "scenes", "renders", "final_png", "final_gif"):
    path = layout[key]
    print(f"{key}: {repo_relative_path(path)}")


## 2. Summary


In [ ]:
combined = load_history(layout["combined"])
sample_id, p_err, f_err, adv = select_best_fourier_advantage_sample(combined)

print("JSON coordinate histories:")
print(f"  {repo_relative_path(layout['pooling'])}")
print(f"  {repo_relative_path(layout['fourier'])}")
print(f"  {repo_relative_path(layout['combined'])}")
print("POV-Ray scene files:")
print(f"  {repo_relative_path(layout['scenes'])}")
print("Rendered figures:")
print(f"  {repo_relative_path(layout['renders'])}")
print(f"  {repo_relative_path(layout['final_png'])}")
print(f"  {repo_relative_path(layout['final_gif'])}")
print(
    "Tracked sample that best demonstrates the Fourier advantage over M8 pooling:"
)
print(
    f"  sample_id={sample_id}  "
    f"final pooled error={p_err:.4f}  "
    f"final Fourier error={f_err:.4f}  "
    f"advantage (pooled−Fourier)={adv:.4f}"
)
